# 00. CONFIGURACIÓN Y VERIFICACIÓN DEL ENTORNO
***
**Descripción:** Este notebook tiene como objetivo establecer la infraestructura de software necesaria, gestionar las credenciales de acceso a datos externos y validar la integridad de las muestras multiespectrales de 14 canales para el proyecto de detección de deslizamientos.

***
## 1. INSTALACIÓN DE DEPENDENCIAS Y CONFIGURACIÓN DE KAGGLE
**Objetivo:** Actualizar la interfaz de línea de comandos de Kaggle, gestionar el despliegue de credenciales API y ejecutar la descarga del dataset Landslide4Sense.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    # Actualización de API de Kaggle
    subprocess.run([sys.executable, "-m", "pip", "install", "kaggle", "--upgrade", "-q"], check=True)

    # Gestión de credenciales
    from google.colab import files
    kaggle_path = Path('/root/.kaggle/kaggle.json')
    
    if not kaggle_path.exists():
        print("Acción requerida: Subir archivo kaggle.json")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            kaggle_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.move('kaggle.json', str(kaggle_path))
            kaggle_path.chmod(0o600)
            print("Estatus: Credenciales configuradas correctamente.")
    
    # Descarga de datos
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", "lxrzp/landslide4sense", "-p", "/content/", "--unzip"],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"Error en descarga: {result.stderr}")
        else:
            print("Estatus: Dataset descargado y descomprimido.")
    except Exception as e:
        print(f"Excepción detectada: {e}")

    # Localización de directorio raíz
    for _root, _dirs, _ in os.walk('/content'):
        if 'TrainData' in _dirs:
            DATA_ROOT_OVERRIDE = _root
            break
else:
    print("Estatus: Ejecución en entorno local detectada.")

***
## 2. VERIFICACIÓN DE ESTRUCTURA Y DIRECTORIOS
**Objetivo:** Confirmar la existencia de las particiones de entrenamiento, validación y prueba, asegurando el conteo correcto de archivos .h5.

In [ ]:
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
else:
    _cwd = Path.cwd()
    _candidates = [_cwd / ".." / "data", _cwd / "data", _cwd.parent]
    DATA_ROOT = next((c.resolve() for c in _candidates if (c / "TrainData" / "img").exists()), None)

if DATA_ROOT is None or not (DATA_ROOT / "TrainData").exists():
    print("Alerta: No se localizó el directorio TrainData.")
else:
    print(f"Directorio raíz: {DATA_ROOT}")
    for part in ["TrainData", "ValidData", "TestData"]:
        img_p = DATA_ROOT / part / "img"
        n = len(list(img_p.glob("*.h5"))) if img_p.exists() else 0
        print(f"  [{'OK' if n > 0 else 'FALLA'}] {part:<10} | Archivos encontrados: {n}")

***
## 3. VALIDACIÓN TÉCNICA Y PRUEBA DE CARGA DE SENSORES
**Objetivo:** Comprobar la disponibilidad de aceleración GPU e inspeccionar una muestra representativa de los canales Sentinel-1 (SAR), Sentinel-2 (Óptico) y DEM (Elevación).

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "segmentation-models-pytorch", "timm", "h5py", "-q"], check=True)

import torch, h5py, numpy as np
import matplotlib.pyplot as plt

print("Especificaciones técnicas:")
print(f"  PyTorch: {torch.__version__}")
print(f"  GPU disponible: {torch.cuda.is_available()}")

try:
    img_files = sorted((DATA_ROOT / "TrainData" / "img").glob("*.h5"))
    if img_files:
        with h5py.File(img_files[0], 'r') as f:
            sample = f['img'][:]
        
        print(f"  Dimensiones del patch: {sample.shape} (14 canales detectados)")

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        channels = [0, 7, 9] 
        names = ['Canal 0: Banda Roja (S2)', 'Canal 7: SAR VV (S1)', 'Canal 9: Modelo de Elevación (DEM)']
        
        for i, (ch, name) in enumerate(zip(channels, names)):
            axes[i].imshow(sample[:,:,ch], cmap='gray' if ch!=9 else 'terrain')
            axes[i].set_title(name)
            axes[i].axis('off')
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f"Error en inspección visual: {e}")